# Atomic Operations with Lua Scripts

This notebook demonstrates `AtomicOperations` for performing multi-step Redis operations atomically using Lua scripts.

## Why Lua Scripts?

- **Atomicity** - Multiple operations execute as a single unit
- **Performance** - Reduces round-trips between client and server
- **Consistency** - No race conditions between steps

## Available Operations

1. **atomic_message_append** - Append message + update metadata atomically
2. **atomic_response_record** - Cache + session + metrics in one operation
3. **atomic_handoff** - Safe agent handoff with distributed locking

In [ ]:
import os
import json
from redis.asyncio import Redis

from redis_openai_agents import AtomicOperations, HandoffInProgressError

# Configuration
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

# Initialize Redis client
client = Redis.from_url(REDIS_URL, decode_responses=True)

# Initialize AtomicOperations
ops = AtomicOperations(client)

print(f"Redis URL: {REDIS_URL}")
print("AtomicOperations initialized")

## 1. Atomic Message Append

Append a message to a session while atomically:
- Incrementing message_count
- Updating timestamp
- Optionally trimming old messages
- Optionally refreshing TTL

In [ ]:
SESSION_KEY = "demo_session:atomic_test"

# Clear any existing session
await client.delete(SESSION_KEY)

# Initialize session with JSON structure
initial_session = {
    "messages": [],
    "metadata": {"message_count": 0},
    "created_at": "2025-01-14T12:00:00Z",
    "updated_at": "2025-01-14T12:00:00Z",
}
await client.json().set(SESSION_KEY, "$", initial_session)
print("Session initialized")

In [ ]:
# Add messages atomically
messages_to_add = [
    {"role": "user", "content": "Hello! How can you help me?"},
    {"role": "assistant", "content": "I can help with Redis and AI agents!"},
    {"role": "user", "content": "Tell me about atomic operations."},
    {"role": "assistant", "content": "Atomic operations ensure consistency..."},
]

print("Adding messages atomically:")
print("=" * 40)

for msg in messages_to_add:
    count = await ops.atomic_message_append(
        session_key=SESSION_KEY,
        message=msg,
        max_messages=10,  # Keep only last 10 messages
        ttl=3600,  # 1 hour TTL
    )
    print(f"  Added [{msg['role']}]: '{msg['content'][:30]}...' -> count: {count}")

In [ ]:
# Verify the session state
session_data = await client.json().get(SESSION_KEY)

print("\nSession State:")
print("=" * 40)
print(f"  Message count: {session_data.get('metadata', {}).get('message_count')}")
print(f"  Messages stored: {len(session_data.get('messages', []))}")
print(f"  Updated at: {session_data.get('updated_at')}")

## 2. Atomic Response Record

Record a response while atomically:
- Storing in cache
- Appending user + assistant messages to session
- Updating metrics (token counts, latency)

In [ ]:
# Reset session for fresh demo
await client.delete(SESSION_KEY)
await client.delete("demo_cache:responses")

initial_session = {
    "messages": [],
    "metadata": {
        "message_count": 0,
        "total_input_tokens": 0,
        "total_output_tokens": 0,
        "total_tokens": 0,
    },
    "created_at": "2025-01-14T12:00:00Z",
    "updated_at": "2025-01-14T12:00:00Z",
}
await client.json().set(SESSION_KEY, "$", initial_session)
print("Session reset for response recording demo")

In [ ]:
# Record a response atomically
result = await ops.atomic_response_record(
    session_key=SESSION_KEY,
    cache_key="demo_cache:responses",
    query_hash="abc123def456",  # Hash of the query for cache lookup
    response="Redis is an in-memory data structure store...",
    user_message={"role": "user", "content": "What is Redis?"},
    assistant_message={"role": "assistant", "content": "Redis is an in-memory data structure store..."},
    latency_ms=150.5,
    input_tokens=15,
    output_tokens=45,
    cache_ttl=3600,
    max_messages=100,
)

print(f"Response recorded: {result}")

In [ ]:
# Verify everything was updated atomically
session_data = await client.json().get(SESSION_KEY)
cache_data = await client.hget("demo_cache:responses", "abc123def456")

print("\nAtomic Update Results:")
print("=" * 40)
meta = session_data.get('metadata', {})
print(f"  Session messages: {meta.get('message_count')}")
print(f"  Total input tokens: {meta.get('total_input_tokens')}")
print(f"  Total output tokens: {meta.get('total_output_tokens')}")
print(f"  Cache entry exists: {cache_data is not None}")

## 3. Atomic Agent Handoff

Perform a safe agent handoff with distributed locking:
- Acquire handoff lock (prevents concurrent handoffs)
- Update current_agent
- Track agents_used
- Store handoff context

In [ ]:
# Reset session for handoff demo
HANDOFF_SESSION = "demo_session:handoff_test"

await client.delete(HANDOFF_SESSION)
await client.delete(f"{HANDOFF_SESSION}:handoff_lock")

initial_session = {
    "messages": [],
    "metadata": {
        "message_count": 0,
        "current_agent": "research_agent",
        "agents_used": ["research_agent"],
    },
    "handoff_context": None,
    "updated_at": "2025-01-14T12:00:00Z",
}
await client.json().set(HANDOFF_SESSION, "$", initial_session)
print("Session ready for handoff demo")
print(f"  Current agent: research_agent")

In [ ]:
# Perform atomic handoff
print("\nPerforming handoff: research_agent -> analysis_agent")
print("=" * 50)

try:
    result = await ops.atomic_handoff(
        session_key=HANDOFF_SESSION,
        from_agent="research_agent",
        to_agent="analysis_agent",
        context={
            "topic": "Redis performance",
            "findings": ["caching improves speed", "use pipelining"],
            "confidence": 0.95,
        },
        lock_ttl=30,
    )
    print(f"  Handoff successful: {result}")
except HandoffInProgressError as e:
    print(f"  Handoff blocked: {e}")

In [ ]:
# Verify handoff state
session_data = await client.json().get(HANDOFF_SESSION)

print("\nSession State After Handoff:")
print("=" * 40)
meta = session_data.get('metadata', {})
print(f"  Current agent: {meta.get('current_agent')}")
print(f"  Agents used: {meta.get('agents_used')}")
print(f"  Handoff context: {session_data.get('handoff_context')}")

In [ ]:
# Release the handoff lock
released = await ops.release_handoff_lock(HANDOFF_SESSION)
print(f"\nHandoff lock released: {released}")

## 4. Concurrent Handoff Protection

Demonstrate how the lock prevents concurrent handoffs.

In [ ]:
import asyncio

async def attempt_handoff(worker_id: str, from_agent: str, to_agent: str) -> str:
    """Attempt a handoff from a worker."""
    try:
        await ops.atomic_handoff(
            session_key=HANDOFF_SESSION,
            from_agent=from_agent,
            to_agent=to_agent,
            context={"worker": worker_id},
            lock_ttl=5,
        )
        return f"{worker_id}: SUCCESS - handed off to {to_agent}"
    except HandoffInProgressError:
        return f"{worker_id}: BLOCKED - another handoff in progress"

# Simulate concurrent handoff attempts
print("Simulating 3 concurrent handoff attempts:")
print("=" * 50)

tasks = [
    attempt_handoff("worker_1", "analysis_agent", "summary_agent"),
    attempt_handoff("worker_2", "analysis_agent", "report_agent"),
    attempt_handoff("worker_3", "analysis_agent", "export_agent"),
]

results = await asyncio.gather(*tasks)
for result in results:
    print(f"  {result}")

print("\nOnly ONE handoff succeeded - the lock prevents race conditions!")

## 5. Message Trimming (Sliding Window)

Demonstrate automatic message trimming to keep only recent messages.

In [ ]:
# Create session with sliding window
TRIM_SESSION = "demo_session:trim_test"

await client.delete(TRIM_SESSION)
await client.json().set(TRIM_SESSION, "$", {
    "messages": [],
    "metadata": {"message_count": 0},
})

# Add 10 messages with max_messages=5
print("Adding 10 messages with max_messages=5 (sliding window):")
print("=" * 50)

for i in range(10):
    count = await ops.atomic_message_append(
        session_key=TRIM_SESSION,
        message={"role": "user", "content": f"Message {i+1}"},
        max_messages=5,  # Keep only last 5
    )
    print(f"  Added message {i+1}, total count: {count}")

# Check what messages remain
session_data = await client.json().get(TRIM_SESSION)
remaining = session_data.get("messages", [])

print(f"\nMessages kept: {len(remaining)}")
for msg in remaining:
    print(f"  - {msg.get('content')}")

## Summary

The `AtomicOperations` class provides:

1. **atomic_message_append** - Thread-safe message addition with metadata updates
2. **atomic_response_record** - Combined cache + session + metrics update
3. **atomic_handoff** - Safe agent handoff with distributed locking

Benefits:
- **Consistency** - No partial updates or race conditions
- **Performance** - Single round-trip for multiple operations
- **Script caching** - EVALSHA for efficient execution

Use this for:
- Multi-step operations that must succeed or fail together
- Preventing race conditions in distributed systems
- High-performance Redis operations

In [ ]:
# Cleanup
await client.delete(SESSION_KEY)
await client.delete(HANDOFF_SESSION)
await client.delete(f"{HANDOFF_SESSION}:handoff_lock")
await client.delete(TRIM_SESSION)
await client.delete("demo_cache:responses")

await client.aclose()
print("Cleanup complete.")